<a href="https://colab.research.google.com/github/Geraldand/2026SpringStat2-Final-Individual-Project/blob/main/notebooks%20/%20Final_Individual_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Data cleaning and value explanned

In [5]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# 1. Load the original dataset
# Ensure 'YRBS_2007 (1).csv' is in the same directory as your script/notebook
df = pd.read_csv('YRBS_2007 (1).csv')

# 2. Define target variables for the research question
# Independent Variable (X)
x_var = 'SafetyConcernsAtSchool'

# Components for the Dependent Variable (Y)
y_components = [
    'WeaponCarrying',
    'GunCarryingPast12Mos',
    'WeaponCarryingAtSchool',
    'PhysicalFighting'
]

# Control / Deep Insight Variables
control_vars = ['WhatIsYourSex', 'SadOrHopeless']

# Combine all target variables for initial filtering
all_target_vars = [x_var] + y_components + control_vars

# 3. Handle Missing Data
# Drop rows with missing values (NaN) within our targeted variables
df_clean = df[all_target_vars].dropna().copy()

# Convert components to numeric, coercing any invalid characters into NaN, then drop them
for col in [x_var] + y_components:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
df_clean = df_clean.dropna()

# 4. Recode Independent Variable (X): Convert categories into actual days
# Original Code Mapping from User Guide:
# 1 = 0 days, 2 = 1 day, 3 = 2 or 3 days, 4 = 4 or 5 days, 5 = 6 or more days
safety_mapping = {
    1: 0.0,
    2: 1.0,
    3: 2.5,
    4: 4.5,
    5: 6.0
}
df_clean['Safety_Days_Missed'] = df_clean[x_var].map(safety_mapping)

# 5. Recode and Aggregate Dependent Variable (Y): Create 'Violence_Risk_Score'
# In YRBS, 1 usually means "0 times / No", and values > 1 mean "Yes (with different frequencies)".
# We binarize these into: Yes = 1, No = 0
for col in y_components:
    df_clean[f'{col}_bin'] = np.where(df_clean[col] > 1, 1, 0)

# Sum the 4 binary behaviors to create a continuous score ranging from 0 to 4
df_clean['Violence_Risk_Score'] = (
    df_clean['WeaponCarrying_bin'] +
    df_clean['GunCarryingPast12Mos_bin'] +
    df_clean['WeaponCarryingAtSchool_bin'] +
    df_clean['PhysicalFighting_bin']
)

# 6. Recode Control Variables (Dummy Encoding)
# Gender: Original code (1 = Female, 2 = Male) -> Convert to Is_Male (1 = Male, 0 = Female)
df_clean['Is_Male'] = np.where(df_clean['WhatIsYourSex'] == 2, 1, 0)

# Depression: Original code (1 = Yes, 2 = No) -> Convert to Is_Sad (1 = Sad/Hopeless, 0 = No)
df_clean['Is_Sad'] = np.where(df_clean['SadOrHopeless'] == 1, 1, 0)

# 7. Filter and Export the Cleaned Dataset
final_features = ['Safety_Days_Missed', 'Violence_Risk_Score', 'Is_Male', 'Is_Sad']
cleaned_df = df_clean[final_features].dropna()

# Save to CSV for project repository compliance
cleaned_df.to_csv('cleaned_data.csv', index=False)
print("✅ Data cleaning complete!")
print(f"Original records: {len(df)} | Cleaned valid records: {len(cleaned_df)}")
print("File successfully saved as 'cleaned_data.csv'\n")

# 8. Run Multiple Linear Regression
# Define predictors (X) and append a constant term for the intercept
X = cleaned_df[['Safety_Days_Missed', 'Is_Male', 'Is_Sad']]
X = sm.add_constant(X)
Y = cleaned_df['Violence_Risk_Score']

# Fit the OLS model
model = sm.OLS(Y, X).fit()

# Display the statistical summary report
print("             ====== Multiple Linear Regression Model Summary ======")
print(model.summary())

✅ Data cleaning complete!
Original records: 14041 | Cleaned valid records: 12997
File successfully saved as 'cleaned_data.csv'

             ====== Multiple Linear Regression Model Summary ======
                             OLS Regression Results                            
Dep. Variable:     Violence_Risk_Score   R-squared:                       0.130
Model:                             OLS   Adj. R-squared:                  0.130
Method:                  Least Squares   F-statistic:                     649.2
Date:                 Thu, 18 Jun 2026   Prob (F-statistic):               0.00
Time:                         14:51:37   Log-Likelihood:                -16899.
No. Observations:                12997   AIC:                         3.381e+04
Df Residuals:                    12993   BIC:                         3.384e+04
Df Model:                            3                                         
Covariance Type:             nonrobust                                         
    